# 05 — Twitch Candidate Generation

Build a first-stage candidate retriever using collaborative filtering.

A production recommender typically does **not** score all 162K streamers with the expensive ranker. Instead:

`full catalog → fast candidate retrieval → expensive ranking model → Top-K`

This notebook builds a lightweight latent-factor retriever from the historical user-streamer interaction matrix and measures whether future-positive streamers appear in the retrieved Top-K.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from time import perf_counter

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
import joblib

PROJECT_DIR = Path("Distributed ML Inference & Ranking")
RAW_PATH = PROJECT_DIR / "data/raw/twitch/100k_a.csv"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/twitch"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "user_id",
    "stream_id",
    "streamer",
    "start_time",
    "stop_time",
]

df = pd.read_csv(RAW_PATH, names=COLUMNS)
df["watch_minutes"] = (
    df["stop_time"] - df["start_time"]
) * 10

df = df[df["watch_minutes"] > 0].copy()

cutoff_test = int(
    np.floor(df["start_time"].quantile(0.85))
)

history = df[df["start_time"] < cutoff_test].copy()
future = df[df["start_time"] >= cutoff_test].copy()

print("History:", history.shape)
print("Future: ", future.shape)

## 1. Create stable user and streamer indices

In [ ]:
user_ids = np.sort(history["user_id"].unique())
streamers = np.sort(history["streamer"].unique())

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(user_ids)
}

streamer_to_idx = {
    streamer: idx
    for idx, streamer in enumerate(streamers)
}

print(f"Users in retriever: {len(user_ids):,}")
print(f"Streamers in retriever: {len(streamers):,}")

## 2. Build sparse implicit-feedback matrix

In [ ]:
pairs = (
    history.groupby(["user_id", "streamer"])
    .agg(
        interactions=("stream_id", "size"),
        watch_minutes=("watch_minutes", "sum"),
    )
    .reset_index()
)

# Confidence-style implicit signal.
pairs["weight"] = (
    np.log1p(pairs["interactions"])
    + 0.25 * np.log1p(pairs["watch_minutes"])
)

rows = pairs["user_id"].map(user_to_idx).to_numpy()
cols = pairs["streamer"].map(streamer_to_idx).to_numpy()
values = pairs["weight"].to_numpy(dtype=np.float32)

R = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_ids), len(streamers)),
)

print("Sparse matrix:", R.shape)
print("Non-zero pairs:", f"{R.nnz:,}")
print("Density:", f"{R.nnz / (R.shape[0] * R.shape[1]):.6%}")

## 3. Train latent-factor candidate retriever

In [ ]:
N_COMPONENTS = 32

svd = TruncatedSVD(
    n_components=N_COMPONENTS,
    n_iter=7,
    random_state=42,
)

start = perf_counter()
user_factors = svd.fit_transform(R).astype(np.float32)
elapsed = perf_counter() - start

item_factors = svd.components_.T.astype(np.float32)

print(f"Training time: {elapsed:.2f} seconds")
print("User factors:", user_factors.shape)
print("Item factors:", item_factors.shape)
print(
    "Explained variance:",
    f"{svd.explained_variance_ratio_.sum():.2%}",
)

## 4. Candidate retrieval function

In [ ]:
def retrieve_candidates(
    user_id,
    top_k=100,
    exclude_seen=True,
):
    if user_id not in user_to_idx:
        return pd.DataFrame(
            columns=["streamer", "retrieval_score"]
        )

    uidx = user_to_idx[user_id]

    scores = (
        user_factors[uidx]
        @ item_factors.T
    )

    if exclude_seen:
        seen_indices = R[uidx].indices
        scores[seen_indices] = -np.inf

    k = min(top_k, len(scores))

    top_idx = np.argpartition(
        scores,
        -k,
    )[-k:]

    top_idx = top_idx[
        np.argsort(scores[top_idx])[::-1]
    ]

    return pd.DataFrame(
        {
            "streamer": streamers[top_idx],
            "retrieval_score": scores[top_idx],
        }
    )


example_user = user_ids[0]
retrieve_candidates(
    example_user,
    top_k=20,
).head(20)

## 5. Build future-positive lookup

In [ ]:
future_positive = (
    future.groupby("user_id")["streamer"]
    .apply(set)
    .to_dict()
)

eligible_users = [
    u for u in user_ids
    if u in future_positive
]

print(
    "Users with history + future interactions:",
    f"{len(eligible_users):,}",
)

## 6. Evaluate candidate Recall@K on a reproducible user sample

In [ ]:
EVAL_USERS = 1000
rng = np.random.default_rng(42)

if len(eligible_users) > EVAL_USERS:
    eval_users = rng.choice(
        eligible_users,
        size=EVAL_USERS,
        replace=False,
    )
else:
    eval_users = np.array(eligible_users)

KS = [50, 100, 500]

results = []

for i, user_id in enumerate(eval_users, start=1):
    truth = {
        s for s in future_positive[user_id]
        if s in streamer_to_idx
    }

    if not truth:
        continue

    retrieved = retrieve_candidates(
        user_id,
        top_k=max(KS),
        exclude_seen=False,
    )["streamer"].tolist()

    row = {
        "user_id": user_id,
        "future_positive_count": len(truth),
    }

    for k in KS:
        hits = len(
            truth.intersection(retrieved[:k])
        )
        row[f"CandidateRecall@{k}"] = (
            hits / len(truth)
        )

    results.append(row)

retrieval_eval = pd.DataFrame(results)

retrieval_eval[
    [c for c in retrieval_eval.columns if c.startswith("CandidateRecall")]
].mean()

## 7. Compare against global popularity retrieval

In [ ]:
popular_streamers = (
    history.groupby("streamer")
    .size()
    .sort_values(ascending=False)
    .index
    .tolist()
)

pop_results = []

for user_id in eval_users:
    truth = {
        s for s in future_positive.get(user_id, set())
        if s in streamer_to_idx
    }

    if not truth:
        continue

    row = {"user_id": user_id}

    for k in KS:
        hits = len(
            truth.intersection(popular_streamers[:k])
        )
        row[f"PopularityRecall@{k}"] = (
            hits / len(truth)
        )

    pop_results.append(row)

pop_eval = pd.DataFrame(pop_results)

pop_eval[
    [c for c in pop_eval.columns if c.startswith("PopularityRecall")]
].mean()

## 8. Save retrieval artifacts

In [ ]:
joblib.dump(
    {
        "svd": svd,
        "user_ids": user_ids,
        "streamers": streamers,
        "user_factors": user_factors,
        "item_factors": item_factors,
        "user_to_idx": user_to_idx,
        "streamer_to_idx": streamer_to_idx,
    },
    ARTIFACT_DIR / "svd_candidate_retriever.joblib",
)

retrieval_eval.to_csv(
    ARTIFACT_DIR / "candidate_retrieval_metrics.csv",
    index=False,
)

print("Saved retriever and candidate-recall results.")

## Next

The candidate retriever is the **first stage**. The ranker is the **second stage**.

In the production Python service we will eventually do:

`user → retrieve ~100–500 candidates → build ranking features → batch inference → sort → Top-K`